In [1]:
import xgboost as xgb
print(xgb.__version__)

3.0.4


In [2]:
import sys, xgboost as xgb
print(sys.executable)        # should point to .../.venv/bin/python
print(xgb.__version__)       # should print 3.0.4
print(xgb.__file__)          # should live under .../.venv/...

/Users/mac/Desktop/US Housing Pulse/.venv/bin/python
3.0.4
/Users/mac/Desktop/US Housing Pulse/.venv/lib/python3.11/site-packages/xgboost/__init__.py


In [3]:
# ==============================================
# 1. Imports
# ==============================================
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import optuna
import mlflow
import mlflow.xgboost

/Users/mac/Desktop/US Housing Pulse/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# ==============================================
# 2. Load processed datasets
# ==============================================
train_df = pd.read_csv("/Users/mac/Desktop/US Housing Pulse/data/processed/feature_engineered_train.csv")
eval_df  = pd.read_csv("/Users/mac/Desktop/US Housing Pulse/data/processed/feature_engineered_eval.csv")


# Define target + features
target = "price"
X_train, y_train = train_df.drop(columns=[target]), train_df[target]
X_eval, y_eval   = eval_df.drop(columns=[target]), eval_df[target]

print("Train shape:", X_train.shape)
print("Eval shape:", X_eval.shape)

Train shape: (576815, 39)
Eval shape: (148448, 39)


In [5]:
# ==============================================
# 3. Define Optuna objective function with MLflow
# ==============================================
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
        "tree_method": "hist",
    }

    with mlflow.start_run(nested=True):
        model = XGBRegressor(**params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_eval)
        rmse = float(np.sqrt(mean_squared_error(y_eval, y_pred)))
        mae = float(mean_absolute_error(y_eval, y_pred))
        r2 = float(r2_score(y_eval, y_pred))

        # Log hyperparameters + metrics
        mlflow.log_params(params)
        mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})

    return rmse

In [6]:
# ==============================================
# 4. Run Optuna study with MLflow
# ==============================================
# Force MLflow to always use the root project mlruns folder
mlflow.set_tracking_uri("/Users/mac/Desktop/US Housing Pulse/mlruns")
mlflow.set_experiment("xgboost_optuna_housing")

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=15)

print("Best params:", study.best_trial.params)

[I 2026-02-10 23:04:43,952] A new study created in memory with name: no-name-41a30d49-8108-4d8d-8589-c8e5df42f794
[I 2026-02-10 23:05:02,625] Trial 0 finished with value: 71841.45667579674 and parameters: {'n_estimators': 800, 'max_depth': 10, 'learning_rate': 0.13638401409726902, 'subsample': 0.7516060313424717, 'colsample_bytree': 0.7037108301823751, 'min_child_weight': 8, 'gamma': 0.21206205431100356, 'reg_alpha': 0.008259507983946493, 'reg_lambda': 5.470022169700459e-06}. Best is trial 0 with value: 71841.45667579674.
[I 2026-02-10 23:05:09,652] Trial 1 finished with value: 75369.78967483084 and parameters: {'n_estimators': 999, 'max_depth': 4, 'learning_rate': 0.023716102498131504, 'subsample': 0.7042970455286541, 'colsample_bytree': 0.7758967284296789, 'min_child_weight': 3, 'gamma': 2.453724583782148, 'reg_alpha': 7.794115307949097e-06, 'reg_lambda': 1.97255376620578}. Best is trial 0 with value: 71841.45667579674.
[I 2026-02-10 23:05:12,907] Trial 2 finished with value: 77517.8

Best params: {'n_estimators': 714, 'max_depth': 9, 'learning_rate': 0.03898097897545712, 'subsample': 0.6140397932651342, 'colsample_bytree': 0.7124809617380705, 'min_child_weight': 1, 'gamma': 4.9727780690432795, 'reg_alpha': 1.307761007223084e-08, 'reg_lambda': 8.413544655099098}


In [7]:
# ==============================================
# 5. Train final model with best params and log to MLflow
# ==============================================
best_params = study.best_trial.params
best_model = XGBRegressor(**best_params)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_eval)

mae = mean_absolute_error(y_eval, y_pred)
rmse = np.sqrt(mean_squared_error(y_eval, y_pred))
r2 = r2_score(y_eval, y_pred)

print("Final tuned model performance:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

# Log final model
with mlflow.start_run(run_name="best_xgboost_model"):
    mlflow.log_params(best_params)
    mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})
    mlflow.xgboost.log_model(best_model, name="model")

Final tuned model performance:
MAE: 30930.155709876006
RMSE: 71473.08016231228
R²: 0.9605229193950698


/Users/mac/Desktop/US Housing Pulse/.venv/lib/python3.11/site-packages/xgboost/sklearn.py:1028: UserWarning: [23:06:50] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  self.get_booster().save_model(fname)
2026/02/10 23:06:53 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/02/10 23:06:53 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
